In [14]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import tensorflow as tf
#from tensorflow.keras import layers, models, losses
#from tensorflow.keras.callbacks import ModelCheckpoint
from keras import layers, models, losses
from keras.models import load_model
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# [LOG] Model Versioning

## Version 1: Tiny-Baseline (Obsolete)
**Data:** 15/05/2026
**Fase:** Upper-Bound Baseline (Test di fattibilità hardware)

### Architettura:
- **Input:** (1, 120, 18) -> [H, W, Channels]
- **Feature Extraction:** - Conv2D (16 filtri, kernel 1x5) + MaxPooling (1x2)
    - SeparableConv2D (32 filtri, kernel 1x3) + MaxPooling (1x2)
- **Output Heads:** - `coords_head`: Dense(8) [Linear] -> X, Y per 4 persone.
    - `mask_head`: Dense(4) [Sigmoid] -> Presenza per 4 persone.

### Statistiche:
- **Parametri Totali:** ~3,500
- **Peso Modello (Float32):** 13.67 KB
- **Peso Stimato (INT8 Quantized):** ~3.5 KB
- **Performance (Epoca 15):** - `val_loss`: 3.79
    - `val_coords_loss`: 3.35 (Errore spaziale medio ~1.83m)
    - `val_mask_loss`: 0.88

## Version 2: Capacità Espansa (Obsolete)
* **Performance:** Errore medio ~1.69m. 
* **Note:** La rete ha smesso di imparare dopo 37 epoche. Mancanza di regolarizzazione (Dropout) e LR fisso.

## Version 3: Architettura "Romana" (Heavy + Callbacks)
**Data:** 31/05/2026
**Fase:** Ottimizzazione Avanzata
* **Architettura:** 12 Layer. Doppie Conv2D(32) -> Doppie SepConv2D(64) -> Conv2D(128) -> Dense(128) + Dropout(0.3) -> Dense(64).
* **Data Pipeline:** `alpha=0.20` (EMA decluttering veloce, come da specifiche). Split dataset con casi complessi (3/4 persone) nel Training. `Batch_Size=8`.
* **Training Hacks:** - `ReduceLROnPlateau`: dimezza il learning rate se la loss si blocca.
    - `EarlyStopping`: ferma l'addestramento se non migliora per 10 epoche e ricarica i pesi migliori.
    - Metrica `RootMeanSquaredError`: legge l'errore spaziale direttamente in Metri.
* **Performance Spaziale:** L'errore medio sulle coordinate è sceso al minimo storico di **~1.56m**.

## [LA SERIE DEI DISASTRI TEORICI: V4 - V6]
*I modelli seguenti rappresentano tentativi falliti di risolvere il problema dell'assegnazione spaziale (Permutation Invariance) e di ignorare i "fantasmi" (persone non presenti). Hanno portato a una serie di Mode Collapse a causa di bug matematici e concettuali.*

## Version 4: Architettura "Imperiale" (Obsolete - Disastroso / Mode Collapse)

### Modifiche Apportate (La Teoria):
- **Spatial Sorting:** Ordinamento forzato dei target da sinistra a destra sull'asse X nel DataGenerator per fornire una regola fissa alla rete (aggirare la *permutation invariance*). **GRAVE ERRORE!!**
- **True Masked MSE:** Modifica della Loss function per moltiplicare l'errore per 0 quando la persona non è presente, smettendo di penalizzare la rete per i "fantasmi".
- **Bilanciamento Loss:** Il peso della `mask_head` è stato portato da 0.5 a 5.0 per costringere l'ottimizzatore a prestare attenzione alla presenza.
- **Custom Metric:** Creata `true_masked_rmse_metres` per calcolare correttamente l'errore in metri gestendo gli array concatenati a 12 valori (8 coords + 4 maschere).

### Statistiche e Limiti Hardware:
- Aumentare indiscriminatamente i filtri a 128 e i layer Densi a 256 ha causato un **esplosione della memoria Flash stimata a 927.42 KB**, superando il limite tassativo di 800 KB dell'ESP32. 

### Il Disastro (Performance):
- Rispetto al modello V3 (e allo split di Davide), **la resa visiva è pessima**. 
- **Sintomo:** Nel visualizzatore, le predizioni (le X rosse) non inseguono minimamente i bersagli. Rimangono immobili, raggruppate e appiccicate in un singolo punto nell'angolo in basso a sinistra della stanza.
- **Causa:** Il layer `GlobalAveragePooling2D`. Calcolando la media matematica su tutta l'ultima mappa di estrazione, ha letteralmente distrutto ogni informazione geometrica. La rete è diventata **cieca**. Non sapendo *dove* guardare, ha applicato un "Mode Collapse": ha imparato a sparare tutte le previsioni nel punto medio statistico per subire la minor penalità possibile dalla Loss.

## Version 5: Architettura "Occhiali" (Obsolete - Mode Collapse Persistente)
* **Modifica Architetturale:** Sostituito il `GlobalAveragePooling2D` con il layer `Flatten()` per ridare alla rete la "vista" geometrica. Usati `MaxPooling2D` aggressivi per mantenere la Flash stimata sotto gli 800 KB (scesa a ~223 KB).
* **Risultato:** Fallimento. Le predizioni continuano a non seguire i bersagli.
* **Diagnosi (Il vero colpevole):** Il problema non era solo il Pooling, ma lo **Spatial Sorting** introdotto nella V4. Poiché le persone si incrociano nella stanza, ordinare le coordinate sull'asse X frame per frame causava il "teletrasporto" dei target da un output all'altro. La rete, non avendo memoria temporale (LSTM), non riusciva a gestire questi sbalzi di gradiente e collassava statisticamente.

## Version 6: Architettura "Tracciante Naturale" (Obsolete - Bug Matematico)
* **Modifica:** Rimosso lo Spatial Sorting, ripristinato l'ordine naturale del dataset. Fixato il bug `axis=1` nella Loss.
* **Risultato:** Fallimento totale e crollo della Binary Accuracy (~50%). Errore fisso a 1.54m misurato magicamente al centro della stanza.
* **Causa 1 (Bug Keras):** Nella funzione custom *True Masked MSE*, mancava il parametro `axis=1` nell'istruzione `tf.reduce_sum()`. Questo fondeva l'errore di un intero batch in un singolo scalare, distruggendo completamente i gradienti frame-per-frame e lobotomizzando la rete.
* **Causa 2:** Problema dell'Assegnazione. Senza ordinamento spaziale, la rete (che non ha memoria temporale LSTM) non sa in quale delle 4 teste di output piazzare le coordinate di una persona in un singolo frame isolato, finendo per sparare al centro statistico per minimizzare la penalità.

## Version 7: Architettura "Ungara" (Breakthrough)
**Fase:** Risoluzione del Problema di Assegnazione
* **Modifica Teorica:** Fusa l'architettura in un'unica testa di output da 12 valori. Introdotta la **Total Hungarian Loss** (Permutation Invariant Training): la rete ora calcola l'errore per tutte le 24 permutazioni possibili e impara solo dall'incrocio geometricamente perfetto, eliminando il Mode Collapse.
* **Risultato:** Le predizioni si sbloccano dal centro della stanza e iniziano a inseguire fisicamente i target reali in movimento.
* **Criticità rilevate (Bug di Misurazione e Metodo):**
    1. **Bug Metrica Spaziale:** Il calcolo dell'errore (0.95m riportati) divideva per il numero di assi, distorcendo la trigonometria (il vero errore euclideo era ~1.34m).
    2. **Bug Metrica Maschera:** Keras misurava l'accuratezza binaria (riportata al 44%) su tutti i 12 output, mescolando coordinate e probabilità.
    3. **Data Split Sbilanciato:** Il set di Validazione ometteva del tutto gli scenari con 2 persone, minando la validità scientifica del test.!!!!


# [GUIDA] Gerarchia del Fine-Tuning per Edge AI (ESP32-S3)

Nel TinyML non possiamo ingrandire la rete a caso, perché siamo limitati da 400KB di RAM e dalla latenza. Le modifiche seguono un ordine di priorità basato sul **Costo Hardware**.

### Livello 1: Costo Hardware ZERO (Modifiche di Addestramento)
Questi parametri non alterano il peso finale del file `.tflite`. Si provano per primi.
* **1. Epoche (`epochs`):** * *Cos'è:* Il tempo di studio. Quante volte la rete vede l'intero dataset.
    * *Quando usarlo:* Se la `val_loss` sta scendendo ma l'addestramento finisce troppo presto (Underfitting).
    * *Effetto:* Permette alla rete di continuare a correggere gli errori.
* **2. Learning Rate (`lr`):**
    * *Cos'è:* La "lunghezza del passo" durante la discesa del gradiente.
    * *Quando usarlo:* Se la Loss salta su e giù in modo impazzito (LR troppo alto) o se non scende per niente fin dall'inizio (LR troppo basso).
    * *Effetto:* Rende l'apprendimento più stabile o più aggressivo.

### Livello 2: Costo Hardware BASSO (Capacità / Larghezza)
* **3. Numero di Filtri (es. da 32 a 64):**
    * *Quando usarlo:* Se la rete è troppo "stupida" per capire le dinamiche della stanza e la Loss si blocca su valori alti (come nella nostra V1).
    * *Effetto:* Aumenta i parametri (Flash) e leggermente la RAM. Dà alla rete più "neuroni" per capire la trigonometria.

### Livello 3: Costo Hardware ALTO (Profondità / Latenza)
* **4. Aggiungere Layer (es. una terza Conv2D):**
    * *Cos'è:* Aggiungere step sequenziali al modello.
    * *Quando usarlo:* Solo se la rete larga non basta per estrarre concetti complessi.
    * *Effetto:* Aumenta drasticamente le operazioni matematiche (MACs). **Aumenta la latenza:** l'ESP32 ci metterà molto più tempo a calcolare ogni singolo frame. Usare con estrema cautela.

In [15]:
# ==============================================================================
# DATA ENGINE V6 (Naturale + Masked Targets, Nessun Sorting)
# ==============================================================================
class EEAIDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, batch_size=8, alpha=0.20, is_training=True):
        self.file_paths = file_paths
        self.batch_size = batch_size
        self.alpha = alpha
        self.is_training = is_training
        if self.is_training:
            np.random.shuffle(self.file_paths)

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.file_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch, y_coords_mask_batch, y_mask_batch = [], [], []

        for file_path in batch_files:
            data = np.load(file_path)
            raw_iq = data['radar_cir_iq']   
            people_xy = data['people_xy']   
            people_mask = data['people_mask'] 
            T = raw_iq.shape[0]             
            
            mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
            mag_reshaped = mag.reshape(T, 1, 120, 18) 
            
            bg = np.copy(mag_reshaped[0])
            decluttered = np.zeros_like(mag_reshaped)
            
            for t in range(T):
                # 1. EMA Decluttering
                bg = self.alpha * mag_reshaped[t] + (1 - self.alpha) * bg
                decluttered[t] = np.abs(mag_reshaped[t] - bg)
            
            # 2. RIMOZIONE SORTING: Prendiamo i dati naturali per evitare il "teletrasporto"
            flat_coords = people_xy.reshape(T, 8)
            
            # 3. Manteniamo il trucco dei 12 valori per la true_masked_mse
            combined_target = np.concatenate([flat_coords, people_mask], axis=1)

            X_batch.append(decluttered)
            y_coords_mask_batch.append(combined_target)
            y_mask_batch.append(people_mask)

        X = np.concatenate(X_batch, axis=0).astype(np.float32)       
        Y_combined = np.concatenate(y_coords_mask_batch, axis=0).astype(np.float32) 
        Y_mask = np.concatenate(y_mask_batch, axis=0).astype(np.float32)   
        
        return X, {"coords_head": Y_combined, "mask_head": Y_mask}

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.file_paths)

# ==============================================================================
# INIZIALIZZAZIONE VARIABILI E SPLIT 
# ==============================================================================
train_indices = [22, 0, 1, 2, 3, 10, 14, 18, 19, 21, 8, 9, 12, 5, 6, 4, 7, 13]
val_indices = [23, 16, 11, 17, 15, 20]

# Ricerca dei file nella cartella corretta
tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# DEFINIZIONE DEL BATCH SIZE (Se il PC fatica con la RAM, abbassalo a 4 o 2)
BATCH_SIZE = 8 

train_gen = EEAIDataGenerator(train_files, batch_size=BATCH_SIZE, alpha=0.20, is_training=True)
val_gen = EEAIDataGenerator(val_files, batch_size=BATCH_SIZE, alpha=0.20, is_training=False)

print(f"Motore V6 pronto: {len(train_files)} file di Train, {len(val_files)} file di Validation.")

Motore V6 pronto: 18 file di Train, 6 file di Validation.


In [16]:
import itertools

# Generiamo in memoria le 24 permutazioni possibili per gli indici [0, 1, 2, 3]
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32) # Shape: [24, 4]

def hungarian_masked_mse(y_true_combined, y_pred_coords):
    # y_true_combined: [Batch, 12] (8 coords + 4 maschere)
    # y_pred_coords: [Batch, 8]

    # 1. Reshape in [Batch, 4_persone, 2_assi] e maschera in [Batch, 4_persone, 1]
    y_true = tf.reshape(y_true_combined[:, :8], (-1, 4, 2))
    y_pred = tf.reshape(y_pred_coords, (-1, 4, 2))
    mask = tf.reshape(y_true_combined[:, 8:], (-1, 4, 1))

    # 2. Generiamo le 24 varianti possibili delle predizioni della rete
    # Questo equivale a "provare tutti gli accoppiamenti"
    y_pred_permuted = tf.gather(y_pred, PERM_INDICES, axis=1) # Shape: [Batch, 24, 4, 2]

    # 3. Espandiamo la verità e la maschera per confrontarle con le 24 varianti
    y_true_exp = tf.expand_dims(y_true, 1) # [Batch, 1, 4, 2]
    mask_exp = tf.expand_dims(mask, 1)     # [Batch, 1, 4, 1]

    # 4. Calcoliamo l'errore quadratico per tutte le 24 permutazioni
    sq_diff = tf.square(y_true_exp - y_pred_permuted)
    masked_sq_diff = sq_diff * mask_exp # Azzeriamo l'errore sui "fantasmi"

    # 5. Sommiamo l'errore di X e Y per le 4 persone -> Otteniamo i 24 "Costi Totali"
    cost_per_perm = tf.reduce_sum(masked_sq_diff, axis=[2, 3]) # Shape: [Batch, 24]

    # 6. HUNGARIAN MATCHING: Troviamo l'accoppiamento perfetto!
    # Scegliamo automaticamente la permutazione che ha generato l'errore minimo
    min_cost = tf.reduce_min(cost_per_perm, axis=1) # Shape: [Batch]

    # 7. Normalizziamo per il numero di coordinate valide
    valid_elements = tf.reduce_sum(mask_exp[:, 0, :, :], axis=[1, 2]) * 2.0 + 1e-6

    return min_cost / valid_elements

def hungarian_masked_rmse_metres(y_true_combined, y_pred_coords):
    # Stessa identica logica della loss, ma applichiamo la radice quadrata alla fine 
    # per avere i metri reali a schermo
    y_true = tf.reshape(y_true_combined[:, :8], (-1, 4, 2))
    y_pred = tf.reshape(y_pred_coords, (-1, 4, 2))
    mask = tf.reshape(y_true_combined[:, 8:], (-1, 4, 1))

    y_pred_permuted = tf.gather(y_pred, PERM_INDICES, axis=1)
    y_true_exp = tf.expand_dims(y_true, 1)
    mask_exp = tf.expand_dims(mask, 1)

    sq_diff = tf.square(y_true_exp - y_pred_permuted)
    masked_sq_diff = sq_diff * mask_exp

    cost_per_perm = tf.reduce_sum(masked_sq_diff, axis=[2, 3])
    min_cost = tf.reduce_min(cost_per_perm, axis=1)
    
    valid_elements = tf.reduce_sum(mask_exp[:, 0, :, :], axis=[1, 2]) * 2.0 + 1e-6

    return tf.sqrt(min_cost / valid_elements)

In [17]:
def embedded_summary(model, input_shape=(1, 120, 18)):
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
    max_layer_ram_kb = 0
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    print("============================================")
    print("   REPORT REQUISITI ESP32-S3 (FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite: 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite: 400 KB)")
    print("============================================\n")

In [18]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V7 "Ungara" (Con Layer ottimizzati V5)
# ==============================================================================
def build_eeai_model_v7(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(64, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(64, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 2))(x) 
    
    x = layers.Conv2D(32, (1, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D((1, 3))(x) 
    
    # FLATTEN: Vista geometrica salvaguardata
    x = layers.Flatten(name="flatten_spatial_map")(x) 
    
    x = layers.Dense(128, activation='relu', name="features_deep")(x)
    x = layers.Dropout(0.3, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_V7_Fixed")

model_v7 = build_eeai_model_v7()

# Questo modello deve rimanere ampiamente sotto gli 800 KB di Flash e i 300 KB di Tensor Arena
embedded_summary(model_v7)

# Compilazione con le funzioni FIXATE
model_v7.compile(
    optimizer='adam',
    loss={"coords_head": hungarian_masked_mse, "mask_head": "binary_crossentropy"}, 
    loss_weights={"coords_head": 1.0, "mask_head": 2.5},
    metrics={
        "coords_head": [hungarian_masked_rmse_metres],
        "mask_head": [tf.keras.metrics.BinaryAccuracy(name="bin_acc")]
    }
)

checkpoint_v7 = ModelCheckpoint("eeai_best_model_v7.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

print("\n--- INIZIO ADDESTRAMENTO V7 UNGARA ---")
history_v7 = model_v7.fit(
    train_gen,                
    validation_data=val_gen,  
    epochs=100,
    callbacks=[checkpoint_v7, reduce_lr, early_stop], 
    verbose=1
)

   REPORT REQUISITI ESP32-S3 (FLOAT32)   
 Memoria FLASH stimata : 223.80 KB  (Limite: 800 KB)
 Memoria SRAM stimata  : ~8.44 KB (Limite: 400 KB)


--- INIZIO ADDESTRAMENTO V7 UNGARA ---


/home/marco/yes/envs/edge_ai_env/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100


I0000 00:00:1780586234.596248   39206 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1780586240.179680   48795 cpu_allocator_impl.cc:82] Allocation of 518400000 exceeds 10% of free system memory.
W0000 00:00:1780586240.551014   40346 cpu_allocator_impl.cc:82] Allocation of 921600000 exceeds 10% of free system memory.
W0000 00:00:1780586240.715186   40346 cpu_allocator_impl.cc:82] Allocation of 460800000 exceeds 10% of free system memory.


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - coords_head_hungarian_masked_rmse_metres: 6.2235 - coords_head_loss: 96.2192 - loss: 109.4730 - mask_head_bin_acc: 0.5351 - mask_head_loss: 3.2100  
Epoch 1: val_loss improved from None to 8.67367, saving model to eeai_best_model_v7.keras
3/3 ━━━━━━━━━━━━━━━━━━━━ 29s 9s/step - coords_head_hungarian_masked_rmse_metres: 5.7314 - coords_head_loss: 68.1600 - loss: 90.8370 - mask_head_bin_acc: 0.5486 - mask_head_loss: 2.7962 - val_coords_head_hungarian_masked_rmse_metres: 1.8136 - val_coords_head_loss: 4.9923 - val_loss: 8.6737 - val_mask_head_bin_acc: 0.4586 - val_mask_head_loss: 1.4725 - learning_rate: 0.0010
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - coords_head_hungarian_masked_rmse_metres: 3.0537 - coords_head_loss: 15.5322 - loss: 21.0235 - mask_head_bin_acc: 0.5182 - mask_head_loss: 1.9718 
Epoch 2: val_loss improved from 8.67367 to 7.33963, saving model to eeai_best_model_v7.keras
3/3 ━━━━━━━━━━━━━━━━━━━━ 26s 7s/step - coords_head_hungari

### Come leggere le Metriche della nostra Hungarian EEAI-Net (V7)

Con l'introduzione dell'**Hungarian Matching** (Permutation Invariant Training), le nostre due teste sono state fuse per calcolare un unico costo globale. Ecco i 4 valori fondamentali che vedrai scorrere sullo schermo e come interpretarli:

1. **hungarian_rmse_metres (L'Errore Spaziale Reale)**
È il traduttore fisico. Indica la distanza media in metri tra le tue predizioni (le X rosse) e le persone reali (i pallini verdi), misurata **dopo** che l'algoritmo ha trovato l'incrocio matematico perfetto. 
*Esempio:* Se vale `1.54`, stai sbagliando in media di 1 metro e mezzo. Più questo numero scende verso lo zero, più le X rosse "inseguiranno" fedelmente i bersagli veri, ignorando i fantasmi.

2. **loss (Il Voto Negativo Globale - Il motore dei gradienti)**
È il "costo" complessivo che la rete usa per correggere i propri errori sui dati di Addestramento. Unisce due punizioni: 
- L'errore balistico di posizione (MSE).
- La penalità se allucina "fantasmi" in slot vuoti (Binary Crossentropy moltiplicata per `2.5`). 
La loss viene calcolata *solo ed esclusivamente* sulla permutazione migliore delle 24 possibili. La rete cerca disperatamente di abbassare questo numero aggiornando i suoi pesi convoluzionali.

3. **val_loss (La Validation Loss - Il Re assoluto dell'addestramento)**
È lo stesso identico calcolo globale della `loss`, ma applicato ai dati che la rete **non ha mai visto** (l'esame di maturità a libro chiuso, es. le finestre 11 e 15). 
*Attenzione:* Questo è il numero più importante di tutti. L'`EarlyStopping` e il `ModelCheckpoint` guardano *esclusivamente* la `val_loss` per capire se il modello sta generalizzando la fisica del radar o se si sta solo imparando a memoria il training set (Overfitting). Finché scende, sei sulla strada giusta.

4. **val_hungarian_rmse_metres (La Prestazione Operativa - Il numero per la Tesi)**
È l'errore spaziale in metri misurato sui dati sconosciuti di validazione. Questo è il dato ufficiale che certificherà la bontà del tuo progetto. Nel tuo report o nella tesi scriverai: *"Il modello ha dimostrato un errore operativo reale di X metri su scenari mai visti durante l'addestramento"*.

In [19]:
# ==============================================================================
# VISUALIZZATORE 3.0 (Anti-Sfarfallio e Ground Truth Fixata)
# ==============================================================================

#file_target = "dataset/data/window_000015.npz"
file_target = "dataset/data/window_000011.npz"
#file_target = "dataset/window_000015.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V6)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Carica il modello migliore passando le funzioni custom
    # (NOTA: usa i nomi esatti delle funzioni che hai usato nel model.compile!)
    model_v7_best = load_model(
        "eeai_best_model_v7.keras",
        custom_objects={
            "hungarian_masked_mse": hungarian_masked_mse,
            "hungarian_masked_rmse_metres": hungarian_masked_rmse_metres
        }
    )

    # Usa esplicitamente il modello MIGLIORE appena caricato!
    preds = model_v7_best.predict(decluttered, verbose=0)
    p_coords = preds[0].reshape(T, 4, 2)
    p_mask = preds[1]
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_title(f"Radar V7 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V6)...
Caricamento dei pesi migliori dal file .keras ...
Dati pronti! Inizializzazione Radar...
